In [ ]:
import urllib.request, json, datetime, math
from collections import defaultdict

# Fetch the data
url = "https://dap.xadi.eu/api/nl/today?markup=0.11&vat=0.21"
print(f"Fetching: {url}")
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=15) as response:
    raw_text = response.read().decode('utf-8')
raw_data = json.loads(raw_text)
print(f"Status: {raw_data.get('status')}, Entries: {len(raw_data.get('data', []))}")

# Show raw JSON structure
print("\n=== RAW JSON STRUCTURE ===")
print(f"Top-level keys: {list(raw_data.keys())}")
print("\nFirst 2 entries:")
for i, entry in enumerate(raw_data.get('data', [])[:2]):
    print(json.dumps(entry, indent=2, default=str))
print(f"\nLast entry:")
print(json.dumps(raw_data['data'][-1], indent=2, default=str))
print(f"\n=== FULL RAW JSON ===")
full_json = json.dumps(raw_data, indent=2, default=str)
print(full_json[:8000])
if len(full_json) > 8000:
    print(f"... [{len(full_json)-8000} more chars]")


In [ ]:
# Parse 15-minute intervals
data_entries = raw_data.get('data', [])
intervals = []
for entry in data_entries:
    ts_str = entry.get('time', '')
    try:
        ts = datetime.datetime.fromisoformat(ts_str.replace('Z', '+00:00'))
    except:
        ts = datetime.datetime.strptime(ts_str[:19], '%Y-%m-%dT%H:%M:%S')
    price = entry.get('price', entry.get('totalPrice', 0))
    hour_str = entry.get('hour', '0:00')
    markup_info = entry.get('markup', {})
    original_price = markup_info.get('originalPrice', price)
    intervals.append({
        'timestamp': ts, 'hour_str': hour_str,
        'hour': int(hour_str.split(':')[0]) if ':' in str(hour_str) else int(hour_str),
        'minute': int(hour_str.split(':')[1]) if ':' in str(hour_str) and len(hour_str.split(':')) > 1 else 0,
        'price': price, 'original_price': original_price,
        'price_mwh': entry.get('priceMwh', price * 1000),
    })
intervals.sort(key=lambda x: x['timestamp'])
print(f"Parsed {len(intervals)} intervals")
print(f"First: {intervals[0]['timestamp']} - {intervals[0]['hour']:02d}:{intervals[0]['minute']:02d} - €{intervals[0]['price']:.4f}/kWh")
print(f"Last:  {intervals[-1]['timestamp']} - {intervals[-1]['hour']:02d}:{intervals[-1]['minute']:02d} - €{intervals[-1]['price']:.4f}/kWh")

# Calculate hourly averages
hourly_data = defaultdict(list)
for iv in intervals:
    hourly_data[iv['hour']].append(iv['price'])
hourly_avg = {h: sum(p)/len(p) for h, p in sorted(hourly_data.items())}

print(f"\n{'='*65}")
print(f"{'Hour (CET)':>12} | {'Avg Price (€/kWh)':>18} | {'# Intervals':>12} | {'Price (ct/kWh)':>14}")
print(f"{'-'*65}")
for hour in range(24):
    if hour in hourly_avg:
        print(f"  {hour:02d}:00       |     €{hourly_avg[hour]:>10.4f}     |     {len(hourly_data[hour]):>4}       |    {hourly_avg[hour]*100:>7.2f} ct")

# Cheapest and most expensive
sorted_hours = sorted(hourly_avg.items(), key=lambda x: x[1])
print(f"\n{'='*50}")
print("TOP 5 CHEAPEST HOURS")
print(f"{'='*50}")
for rank, (hour, price) in enumerate(sorted_hours[:5], 1):
    neg = " ⚡ NEGATIVE!" if price < 0 else ""
    print(f"  #{rank}: {hour:02d}:00 → €{price:.4f}/kWh ({price*100:.2f} ct){neg}")
print(f"\n{'='*50}")
print("TOP 5 MOST EXPENSIVE HOURS")
print(f"{'='*50}")
for rank, (hour, price) in enumerate(sorted_hours[-5:][::-1], 1):
    print(f"  #{rank}: {hour:02d}:00 → €{price:.4f}/kWh ({price*100:.2f} ct)")

# Statistics
all_prices = [iv['price'] for iv in intervals]
hourly_prices_list = list(hourly_avg.values())
min_hourly = min(hourly_prices_list)
max_hourly = max(hourly_prices_list)
spread = max_hourly - min_hourly
mean_price = sum(all_prices) / len(all_prices)
sorted_prices = sorted(all_prices)
median_price = sorted_prices[len(sorted_prices) // 2]
std_dev = (sum((p - mean_price) ** 2 for p in all_prices) / len(all_prices)) ** 0.5
neg_hours = sum(1 for h, p in hourly_avg.items() if p < 0)
neg_intervals = sum(1 for iv in intervals if iv['price'] < 0)

print(f"\n{'='*55}")
print("PRICE STATISTICS")
print(f"{'='*55}")
print(f"  Min hourly avg:       €{min_hourly:.4f}/kWh ({min_hourly*100:.2f} ct)")
print(f"  Max hourly avg:       €{max_hourly:.4f}/kWh ({max_hourly*100:.2f} ct)")
print(f"  Price spread:         €{spread:.4f}/kWh ({spread*100:.2f} ct)")
print(f"  Mean (all intervals): €{mean_price:.4f}/kWh ({mean_price*100:.2f} ct)")
print(f"  Median:               €{median_price:.4f}/kWh ({median_price*100:.2f} ct)")
print(f"  Std deviation:        €{std_dev:.4f}/kWh ({std_dev*100:.2f} ct)")
print(f"  Negative price hours: {neg_hours}")
print(f"  Negative 15-min intervals: {neg_intervals}")


In [ ]:
# Battery specs - 1 unit
charge_power = 0.8    # kW
discharge_power = 0.8  # kW
capacity = 2.7         # kWh
efficiency = 0.75
charge_time = capacity / charge_power  # 3.375 hours
usable_discharge = capacity * efficiency  # 2.025 kWh
discharge_time = usable_discharge / discharge_power  # 2.53125 hours

print(f"{'='*55}")
print("BATTERY SPECS — 1 UNIT")
print(f"{'='*55}")
print(f"  Charge power:          {charge_power*1000:.0f} W ({charge_power} kW)")
print(f"  Battery capacity:      {capacity} kWh")
print(f"  Round-trip efficiency:  {efficiency*100:.0f}%")
print(f"  Charge time (full):    {charge_time:.3f}h ({charge_time*60:.0f} min)")
print(f"  Usable discharge:      {usable_discharge:.3f} kWh")
print(f"  Discharge time:        {discharge_time:.3f}h ({discharge_time*60:.0f} min)")
print(f"  Break-even: discharge > charge_price / {efficiency}")

# Optimal strategy: find best non-overlapping charge/discharge windows
hours_needed_charge = math.ceil(charge_time)    # 4
hours_needed_discharge = math.ceil(discharge_time)  # 3

best_profit = float('-inf')
best_combo = None

for cs in range(24 - hours_needed_charge + 1):
    ch = list(range(cs, cs + hours_needed_charge))
    if not all(h in hourly_avg for h in ch):
        continue
    # Calculate charge cost
    c_cost = 0; e_rem = capacity
    for h in ch:
        e = min(charge_power, e_rem)
        c_cost += hourly_avg[h] * e
        e_rem -= e
        if e_rem <= 0: break
    
    for ds in range(24 - hours_needed_discharge + 1):
        dh = list(range(ds, ds + hours_needed_discharge))
        if not all(h in hourly_avg for h in dh):
            continue
        if set(ch) & set(dh):
            continue
        # Calculate discharge revenue
        d_rev = 0; e_rem = usable_discharge
        for h in dh:
            e = min(discharge_power, e_rem)
            d_rev += hourly_avg[h] * e
            e_rem -= e
            if e_rem <= 0: break
        profit = d_rev - c_cost
        if profit > best_profit:
            best_profit = profit
            best_combo = (ch, dh, c_cost, d_rev)

charge_hours, discharge_hours, total_charge_cost, total_discharge_revenue = best_combo
net_profit = total_discharge_revenue - total_charge_cost

# Assign labels
hour_labels = {}
for h in range(24):
    if h in charge_hours: hour_labels[h] = 'CHARGE'
    elif h in discharge_hours: hour_labels[h] = 'DISCHARGE'
    else: hour_labels[h] = 'idle'

print(f"\n{'='*60}")
print("OPTIMAL STRATEGY (1 UNIT)")
print(f"{'='*60}")
print(f"\n📥 CHARGE: {charge_hours[0]:02d}:00 – {charge_hours[-1]+1:02d}:00")
e_rem = capacity
for h in charge_hours:
    e = min(charge_power, e_rem)
    if e <= 0: break
    print(f"   {h:02d}:00: {e:.3f} kWh × €{hourly_avg[h]:.4f} = €{hourly_avg[h]*e:.4f}")
    e_rem -= e

print(f"\n📤 DISCHARGE: {discharge_hours[0]:02d}:00 – {discharge_hours[-1]+1:02d}:00")
e_rem = usable_discharge
for h in discharge_hours:
    e = min(discharge_power, e_rem)
    if e <= 0: break
    print(f"   {h:02d}:00: {e:.3f} kWh × €{hourly_avg[h]:.4f} = €{hourly_avg[h]*e:.4f}")
    e_rem -= e

energy_charged = capacity
energy_discharged = usable_discharge
avg_charge = total_charge_cost / energy_charged
avg_discharge = total_discharge_revenue / energy_discharged
breakeven_threshold = avg_charge / efficiency

print(f"\n{'='*60}")
print("PROFIT BREAKDOWN")
print(f"{'='*60}")
print(f"  Total charge cost:        €{total_charge_cost:.4f}")
print(f"  Total discharge revenue:  €{total_discharge_revenue:.4f}")
print(f"  Energy charged:           {energy_charged:.3f} kWh")
print(f"  Energy discharged:        {energy_discharged:.3f} kWh")
print(f"  Energy lost (25%):        {energy_charged - energy_discharged:.3f} kWh")
print(f"  Avg charge price:         €{avg_charge:.4f}/kWh")
print(f"  Avg discharge price:      €{avg_discharge:.4f}/kWh")
print(f"  Break-even threshold:     €{breakeven_threshold:.4f}/kWh")
print(f"  💰 NET PROFIT:            €{net_profit:.4f}/cycle")
if net_profit > 0:
    print(f"  ✅ PROFITABLE — Annual est: €{net_profit*365:.2f}")
else:
    print(f"  ⚠ NOT PROFITABLE — spread too small for 75% efficiency")

# 4-unit comparison
scale = 4
print(f"\n{'='*65}")
print("4-UNIT COMPARISON")
print(f"{'='*65}")
print(f"\n  {'Parameter':<28} {'1 Unit':>12} {'4 Units':>12}")
print(f"  {'-'*55}")
print(f"  {'Charge power':<28} {charge_power*1000:>10.0f} W  {charge_power*scale*1000:>10.0f} W")
print(f"  {'Battery capacity':<28} {capacity:>10.1f} kWh{capacity*scale:>10.1f} kWh")
print(f"  {'Usable discharge':<28} {usable_discharge:>10.3f} kWh{usable_discharge*scale:>10.3f} kWh")
print(f"  {'Charge time':<28} {charge_time:>10.3f} h  {charge_time:>10.3f} h")
print(f"  {'Charging cost':<28} €{total_charge_cost:>10.4f}  €{total_charge_cost*scale:>10.4f}")
print(f"  {'Discharge revenue':<28} €{total_discharge_revenue:>10.4f}  €{total_discharge_revenue*scale:>10.4f}")
print(f"  {'NET PROFIT / cycle':<28} €{net_profit:>10.4f}  €{net_profit*scale:>10.4f}")
print(f"  {'Annual (365 cycles)':<28} €{net_profit*365:>10.2f}  €{net_profit*scale*365:>10.2f}")
print(f"\n  Same hours, everything scales 4× linearly.")

# Full schedule
print(f"\n{'='*80}")
print("FULL HOURLY SCHEDULE")
print(f"{'='*80}")
for h in range(24):
    if h in hourly_avg:
        label = hour_labels.get(h, 'idle')
        m = "🟢" if label == 'CHARGE' else ("🔴" if label == 'DISCHARGE' else "⚪")
        bar_w = 40
        if max_hourly != min_hourly:
            bar_len = int((hourly_avg[h] - min_hourly) / (max_hourly - min_hourly) * bar_w)
        else:
            bar_len = bar_w // 2
        bar = '█' * bar_len
        print(f"  {m} {h:02d}:00 |{bar:<{bar_w}}| €{hourly_avg[h]:.4f} {'← '+label if label != 'idle' else ''}")

print(f"\n  Legend: 🟢 = Charging | 🔴 = Discharging | ⚪ = Idle")
print(f"  Spread: €{spread:.4f} | Profit/cycle (1u): €{net_profit:.4f} | (4u): €{net_profit*4:.4f}")
